## Import data

In [10]:
from pathlib import Path


job_ads = []
for file in Path("data").glob("ads*.txt"):
    with open(file, "r", encoding="utf-8") as f:
        job_ads.append(f.read())

print(f"Loaded {len(job_ads)} job ads")
print(job_ads[0][:500])

Loaded 3 job ads
At NOBA Bank Group, we are unlocking new possibilities as we enter an exciting phase with great untapped potential! As the Engineering Manager for one of our Software Engineering teams, you will play a crucial role in both leading daily deliveries and engaging in strategic discussions with Product and Technology leadership. This is your opportunity to shape our technology landscape and create real business impact.

About the role:

We are looking for an Engineering Manager to join our Software E


## Data models

In [19]:
from pydantic import BaseModel, Field 

class JobsAdSummary(BaseModel):
    job_title: str 
    description: str
    summary: str
    responsibilities: list[str] 
    words_in_article: int



## Create agent

In [20]:
from pydantic_ai import Agent

job_ads_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    instructions=""" 
You summarize job ads into structured data. 

Extract:
- job_title
- description
- summary
- responsibilities
- words_in_article

Rules:
- Only extract information clearly supported by the ad.
- Keep summary short and factual.
- Responsibilitites should be a list of main responsibilitites.
- words_in_article should be the approxiamate number of words in the full input article. 
- Use the same language as the text when appropriate. 
 
"""
)

async def summarize_job_ad(ad: str) -> JobsAdSummary:
    result = await job_ads_agent.run(ad, output_type=JobsAdSummary)
    return result.output

summaries = []
for ad in job_ads:
    summary = await summarize_job_ad(ad)
    summaries.append(summary)

summaries



[JobsAdSummary(job_title='Engineering Manager', description='Lead daily software deliveries and engage in strategic discussions with Product and Technology leadership to shape the technology landscape and create business impact.', summary="Engineering Manager role in Stockholm leading a team of 5 engineers for Nordax brand's customer onboarding, focusing on software development across the full stack, collaborating with Product and Tech leadership to drive technical decisions and team performance.", responsibilities=['Lead and mentor a team of engineers', 'Drive cross-team collaboration and initiatives', 'Ensure business objectives and project deadlines are met', 'Provide technical leadership in software development decisions', 'Uphold best practices in development processes', 'Contribute to recruitment and organizational growth'], words_in_article=460),
 JobsAdSummary(job_title='Data Center Engineering Operations Technician', description='We are currently seeking a Data Center Engineer

In [22]:
from pathlib import Path

output_dir = Path("markdown_exports")
output_dir.mkdir(exist_ok=True)

for i, (ad, summary) in enumerate(zip(job_ads, summaries), start=1):
    md_content = f"""# Job Ad {i}
## Orginal ad

{ad}

## Summary

- **Job title:** {summary.job_title}
- **Description:** {summary.description}
- **Job title:** {summary.summary}
- **Responsibilities:**
"""
    for responsibility in summary.responsibilities:
        md_content += f" -{responsibility}\n"
    
    md_content += f"\n- **Words in article:** {summary.words_in_article}\n"

    with open(output_dir / f"job_ad_{i}.md", "w", encoding="utf-8") as f:
        f.write(md_content)